# 03-3 — Native Tool Architecture: `BaseTool`

Demonstrates the low-level `BaseTool` / `ToolResult` API that underpins all tools.

| Topic | Description |
|---|---|
| `BaseTool` subclass | Define a tool with `input_schema` + `execute()` |
| `get_mcp_schema()` | MCP wire format (`name`, `description`, `inputSchema`) |
| `get_openai_schema()` | OpenAI function-calling format (`type: "function"`, ...) |
| `ToolResult` | Typed result with `content` blocks and optional `app_data` |
| `ToolExecutionResultMessage` | Canonical message wrapping a tool result |
| Risk & HITL | Class-level `ToolRisk` + `HitlMode` strategy |

**No external services needed** — all cells are offline.


In [1]:
import json
from ravi.kernel.tools.base_tool import BaseTool, ToolResult, ToolRisk, HitlMode
from ravi.kernel.messages.client_messages import ToolExecutionResultMessage
from ravi.kernel.messages.content import TextBlock


---
## 1. Define a tool by subclassing `BaseTool`

- Constructor: pass `name`, `description`, `input_schema` (JSON Schema object)
- `execute()` must use **keyword-only** params (`*,`) matching the schema properties
- Add `# type: ignore[override]` on `execute()` — BaseTool's signature is intentionally broad


In [2]:
class AddTool(BaseTool):
    """Add two numbers — minimal BaseTool example."""

    def __init__(self):
        super().__init__(
            name="add_numbers",
            description="Add two numbers together and return the result.",
            input_schema={
                "type": "object",
                "properties": {
                    "a": {"type": "number", "description": "First number"},
                    "b": {"type": "number", "description": "Second number"},
                },
                "required": ["a", "b"],
            },
        )

    async def execute(self, *, a: float, b: float) -> ToolResult:  # type: ignore[override]
        """BaseTool.run() validates inputs before calling this."""
        return ToolResult(
            content=[TextBlock(text=f"{a} + {b} = {a + b}")],
            app_data={"result": a + b},  # structured payload for the agent
        )


tool = AddTool()
print(f"Tool name : {tool.name}")
print(f"Risk      : {tool.risk}")
print(f"HITL mode : {tool.hitl_mode}")


Tool name : add_numbers
Risk      : ToolRisk.SAFE
HITL mode : HitlMode.BLOCKING


---
## 2. Dual schema output — MCP and OpenAI formats


In [3]:
print("=== MCP wire format ===")
print(json.dumps(tool.get_mcp_schema(), indent=2))

print("\n=== OpenAI function-calling format ===")
print(json.dumps(tool.get_openai_schema(), indent=2))


=== MCP wire format ===
{
  "name": "add_numbers",
  "description": "Add two numbers together and return the result.",
  "inputSchema": {
    "type": "object",
    "properties": {
      "a": {
        "type": "number",
        "description": "First number"
      },
      "b": {
        "type": "number",
        "description": "Second number"
      }
    },
    "required": [
      "a",
      "b"
    ]
  }
}

=== OpenAI function-calling format ===
{
  "type": "function",
  "function": {
    "name": "add_numbers",
    "description": "Add two numbers together and return the result.",
    "parameters": {
      "type": "object",
      "properties": {
        "a": {
          "type": "number",
          "description": "First number"
        },
        "b": {
          "type": "number",
          "description": "Second number"
        }
      },
      "required": [
        "a",
        "b"
      ]
    },
    "strict": true
  }
}


---
## 3. Execute the tool + build a `ToolExecutionResultMessage`


In [4]:
# Execute the tool directly (BaseTool.run() validates schema first)
result = await tool.run(a=5, b=7)
print(f"ToolResult content : {result.content[0].text}")
print(f"ToolResult app_data: {result.app_data}")
print(f"is_error           : {result.is_error}")

# Wrap in a ToolExecutionResultMessage (what the agent loop sends back to the LLM)
msg = ToolExecutionResultMessage.from_tool_result(
    tool_result=result,
    tool_call_id="call_abc123",
    tool_name="add_numbers",
)
print(f"\nmsg.tool_call_id : {msg.tool_call_id}")
print(f"msg.name         : {msg.name}")
print(f"msg.content[0]   : {msg.content[0].text}")


ToolResult content : 5 + 7 = 12
ToolResult app_data: {'result': 12}
is_error           : False

msg.tool_call_id : call_abc123
msg.name         : add_numbers
msg.content[0]   : 5 + 7 = 12


---
## 4. Risk & HITL strategy — class-level attributes

Tools that perform dangerous operations (send email, write files, deploy code) should
set `risk` and `hitl_mode` as **class-level attributes** — not in `__init__`.

| `ToolRisk` | `HitlMode` | Behaviour |
|---|---|---|
| `SAFE` | `NONE` | Auto-execute, no approval |
| `MODERATE` | `NOTIFY` | Execute + post-hoc notification |
| `HIGH` | `NON_BLOCKING` | Execute but log for review |
| `CRITICAL` | `BLOCKING` | **Pause** and wait for human approval |


In [ ]:
class CriticalDeployTool(BaseTool):
    # ← Strategy pattern: class-level, NOT in __init__
    risk = ToolRisk.CRITICAL
    hitl_mode = HitlMode.BLOCKING

    def __init__(self):
        super().__init__(
            name="deploy_to_production",
            description="Deploy a service to production — requires human approval",
            input_schema={
                "type": "object",
                "properties": {
                    "service": {"type": "string"},
                    "version": {"type": "string"},
                },
                "required": ["service", "version"],
            },
        )

    async def execute(self, *, service: str, version: str) -> ToolResult:  # type: ignore[override]
        return ToolResult(content=[TextBlock(text=f"Deployed {service}@{version}")])


deploy_tool = CriticalDeployTool()
print(f"risk      : {deploy_tool.risk}")
print(f"hitl_mode : {deploy_tool.hitl_mode}")
print(f"Will block for human approval: {deploy_tool.hitl_mode == HitlMode.BLOCKING}")


risk      : ToolRisk.CRITICAL
hitl_mode : HitlMode.BLOCKING
Will block for human approval: True


: 